Please read carefully the following lines before proceeding with the exercise:
---
#### 1. Make your own copy of this notebook to your Drive. Work on that copy instead.
#### 2. Make sure you add a shortcut of the [CV_6-CNN](https://drive.google.com/drive/folders/1mwSMGZUfw-CgGwA9NWBdnFCoQWjEPOPO?usp=sharing) shared folder to your Drive.
#### 3. Any files you wish to create can only be saved to your Drive, not in the shared folder. For that reason the second cell makes a copy of the shared folder to your drive ('copy_CV_6-CNN').
#### 4. Don't forget to change your Runtime Type to "GPU".
#### 5. Remember that each time you close this tab, your Google Drive is unmounted and all variables are lost.


Have fun :)



In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp -r '/content/drive/MyDrive/CV_6-CNN' '/content/drive/MyDrive/copy_CV_6-CNN'

In [41]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import matplotlib.pyplot as plt
plt.style.use('dark_background')
plt.rcParams['image.cmap'] = 'gist_gray'
from torch.utils.data import DataLoader
from torchvision import models
from torchvision import transforms
from matplotlib.patches import Rectangle
from torchvision.datasets import CocoDetection
device = torch.device('cuda:0')

Dataset

In [34]:
dataset_val = CocoDetection(root='/content/drive/MyDrive/CV_6-CNN/val2017/',
                            annFile='/content/drive/MyDrive/CV_6-CNN/annotations_trainval2017/annotations/instances_val2017.json',
                            transform=transforms.Compose([transforms.ToTensor(),transforms.Resize((320,213))]))


loading annotations into memory...
Done (t=0.73s)
creating index...
index created!


Dataloader

In [35]:
val_dataloader = DataLoader(dataset_val, batch_size=1, shuffle=False, num_workers=0)

Faster-RCNN with ResNet50 backbone CNN and FPN

In [ ]:
model = models.detection.fasterrcnn_resnet50_fpn(pretrained=True,num_classes=91).to(device)

SSD with VGG16 backbone CNN

In [ ]:
model = models.detection.ssd300_vgg16(pretrained=True,num_classes=91).to(device)

In [ ]:
from tqdm import tqdm # fancy and unnecessary library to add loading bar for the loop

model.eval() # switch to evaluation mode, 

all_detections = []
all_targets = []
for imgs_batch,targets_batch in tqdm(val_dataloader):
    # Move imgs_batch and targets_batch to device
    imgs_batch = [imgs.to(device) for imgs in imgs_batch]
    
    targets_batch = [targets for targets in targets_batch]

    # Transform targets to COCO format.
    new_targets_batch = []
    targets_batch = [targets_batch]

    for targets in targets_batch:   
        new_targets = {'boxes':[],'labels':[]}
        for t in targets:
            x,y,w,h = [s for s in t['bbox']]
            new_targets['boxes'].append(torch.Tensor([x, y, x+w, y+h]))
            new_targets['labels'].append(t['category_id'])
        if len(new_targets['boxes']) > 0:
            new_targets['boxes'] = torch.stack(new_targets['boxes'], dim=0)
        new_targets['labels'] = torch.Tensor(new_targets['labels']).long()
        new_targets_batch.append(new_targets)
    targets_batch = new_targets_batch

    # Forward pass
    with torch.no_grad():
        detections_batch = model(imgs_batch)

    del imgs_batch 
    

    # Keep detections and targets to compare
    all_targets.append(targets_batch)
    all_detections.append(detections_batch)

torch.save(all_detections, '/content/drive/MyDrive/copy_CV_6-CNN/all_detections_faster.pth')
torch.save(all_targets,'/content/drive/MyDrive/copy_CV_6-CNN/all_targets_faster.pth')